# Inheco ODTC

```{device-card} inheco-odtc
```

| Property | Value |
|---|---|
| Connection | Ethernet, SiLA 1 SOAP; device port 8080 |
| Models | 96-well and 384-well |
| Block temperature | 4–99 °C |
| Lid temperature | 30–110 °C (96); 30–115 °C (384) |
| Units | °C, seconds, microliters |

**Hardware verification required for this implementation on both variants.** Offline tests cover
protocol compilation and simulated communication; they do not establish thermal accuracy or safe
robotic placement. `setup()` logs a warning until a verified hardware run is reported.

## Connection and physical setup

Connect the ODTC controller to Ethernet and power, following the manufacturer's operating manual.
Configure an address reachable from this computer. The instrument sends completion and temperature
callbacks to the computer: allow the selected `client_ip` and `callback_port` through its firewall.
The default callback port is chosen automatically. Supply a fixed port if firewall rules require it.

The driver opens no connection during construction. `plate_holder` represents the plate location;
its zero child offset is **not a measured loading position**. Calibrate its placement before robotic
transfers. Load compatible labware manually for an initial hardware check.

The controls below move and heat real hardware. Run them only when the instrument and labware are ready.

In [ ]:
from pylabrobot.inheco import ODTC
from pylabrobot.inheco.odtc import (
    ODTCProtocol, ODTCStage, ODTCStep, compile_protocol, method_set_to_xml, ODTCMethodSet,
)

odtc = ODTC(
    name="odtc",
    odtc_ip="192.168.1.50",  # use the configured instrument address
    variant=96,
    # client_ip="192.168.1.10",  # computer interface reachable from the ODTC
    # callback_port=8081,
)

## Set up

Set up the HTTP connection and callback listener, register the listener, and initialize the device.
Setup refuses to reset a busy or faulted instrument. Calling it again on an established session leaves
the session intact. `simulation_mode=True` selects the instrument firmware's simulation mode; it is
not an offline simulator.

In [ ]:
await odtc.setup()

## Open the drawer

Open and wait for command completion. Repeating an already confirmed open or close does not issue
another movement. The firmware provides no door-position query: `door_open` is the last
command-confirmed state, or `None` after connection, a failed movement, or a fault.

In [ ]:
await odtc.open_door()

## Load the plate, then close

Place the plate on the ODTC mount before running this cell.

In [ ]:
await odtc.close_door()

## Read temperatures

Read all eight sensors in degrees C. This operation is also available while a method is running.

In [ ]:
temperatures = await odtc.request_temperatures()
print(temperatures)

## Define a temperature profile

`ODTCStep` holds a target temperature and duration. `ODTCStage` repeats an ordered sequence of steps
or nested stages. Temperatures, durations, and ramp rates are checked before upload. Native methods
are limited to 100 steps; deeply nested loops sharing an endpoint may need expansion.

Lid temperature and maximum sample volume are supplied per run. An explicit step lid temperature
or ramp rate overrides the run's lid default or the variant's transition rate.

In [ ]:
protocol = ODTCProtocol(stages=[
    ODTCStage([ODTCStep(95, duration=300)]),
    ODTCStage([
        ODTCStep(95, duration=30),
        ODTCStep(55, duration=30),
        ODTCStep(72, duration=60),
    ], repeats=30),
    ODTCStage([ODTCStep(72, duration=600), ODTCStep(4, duration=30)]),
])

## Inspect the native method without hardware

Compilation is pure Python. The volume selects the device's compensation range: below 30 µL,
30–below 75 µL, or 75–100 µL. The volume must be positive and no greater than 100 µL.

Automatic empirical overshoot is **disabled by default**: the coefficients have not been validated
against a manufacturer reference or this implementation's hardware runs. `apply_overshoot=True`
opts in with a warning. A loop entry with different incoming temperatures requires explicit
`ODTCOvershoot` settings because a single calculated excursion would be ambiguous. Its `temperature`
is an excursion magnitude, not an absolute target. Explicit overshoot and PID settings are retained
in native XML round trips. Validate thermal compensation on the actual model and labware.

In [ ]:
method = compile_protocol(protocol, variant=96, volume=25, lid_temperature=105)
print(method_set_to_xml(ODTCMethodSet(methods=[method])))

## Run: prepare, execute, and wait

`run_protocol` uploads the method, runs a matching block/lid premethod when necessary, and waits for
completion. Preparation can take several minutes. It waits for firmware completion rather than
merely observing the target on a temperature sensor. Compatible conditions retained from a completed
premethod or post-heated method can be reused within the session.

The firmware executes timed temperature steps because its command interface has no direct
block-setpoint primitive. PLR performs compilation and sequencing.

`post_heating=True` keeps regulation at the final step after completion. Pass `False` to disable it.
`timeout` limits each execution wait; a timeout **does not stop the instrument**.

In [ ]:
await odtc.run_protocol(protocol, volume=25, lid_temperature=105, post_heating=True)

## Read measured progress

The latest callback provides elapsed time and target, block, and lid temperatures. `None` means no
sample has arrived for this execution. Step numbers, cycle counts, and estimated remaining time are
not reported as measured device state.

In [ ]:
progress = await odtc.request_progress()
print(progress)

## Stop temperature control

This stops an active method or regulation remaining after a method or premethod. It remains available
while `run_protocol()` is awaiting completion. An interrupted run raises `ODTCProtocolStopped`.
To run concurrently with other work, create an asyncio task for `run_protocol`; canceling that task
only cancels the wait. Call `stop_protocol()` to stop the hardware. A later `wait_for_completion()`
still observes completion errors.

In [ ]:
await odtc.stop_protocol()

## Prepare independently

Use explicit preparation when the block and lid should be ready before loading a plate. The same
preparation is included in `run_protocol`. Temperatures are per-operation arguments.

In [ ]:
await odtc.prepare(temperature=95, lid_temperature=105)

## Store a named method

Compile with a persistent name, then upload. Existing names require `overwrite=True`. The
`plr_currentProtocol` and `plr_currentPreMethod` slots are reserved for the driver's working methods;
unnamed runs replace only these slots. Uploads never clear the instrument's method library.

In [ ]:
stored_method = compile_protocol(
    protocol, name="StandardPCR", variant=96, volume=25, lid_temperature=105,
)
await odtc.upload_protocol(stored_method)

## Retrieve methods

Downloaded `ODTCMethod` objects retain native step numbers, loop targets/counts, independent overshoot
rates, PID sets, and metadata. They can be inspected or edited using `dataclasses.replace`, then
uploaded explicitly. `request_method_set()` includes premethods too.

In [ ]:
stored = await odtc.request_protocol("StandardPCR")
print(stored)

## Run a stored method

Use its stored volume, lid, and ramp settings. PLR prepares matching initial thermal conditions
before executing it.

In [ ]:
await odtc.run_stored_protocol("StandardPCR")

## Disconnect

`stop()` stops temperature control and closes the HTTP connection and callback server. Transport
cleanup also runs if the stop command fails; the exception still reports that the device did not
confirm stopping.

In [ ]:
await odtc.stop()

## Variant verification and protocol references

Choose `variant=384` for a 384-well unit. Its current compiler limits are 5 °C/s heating and 115 °C lid,
compared with 4.4 °C/s and 110 °C for the 96-well unit. Both use a 2.2 °C/s cooling limit. The variant
codes, plate types, volume compensation, and limits need confirmation on each hardware variant;
passing mock tests is not hardware validation.

The wire format, method storage, premethod sequence, callback sample units, and return codes are
covered by the [INHECO Firmware Command Set v1.06](https://github.com/user-attachments/files/26010799/TD_SILA_FWCommandSet.25210.pdf)
(sections 3, 5, 6, 7.3, 9, and 10). This 2016 document does not establish which firmware revision is
installed on a particular device. Record the instrument and firmware version when reporting a
[hardware verification run](../../needs-testing.md). The tests use its example method and data series
as fixtures, alongside simulated SOAP exchanges. Thermal overshoot coefficients are not established
by that command-set document.